# Choosing and replacing Python components


In [1]:
from pathlib import Path
import ast
import re
import sys
import tomllib

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from matplotlib.patches import FancyArrowPatch, FancyBboxPatch
import numpy as np
import pandas as pd
from IPython.display import HTML, display as _display


def display(value):
    module = value.__class__.__module__
    if module.startswith("pandas") and hasattr(value, "_repr_html_"):
        html = value._repr_html_()
        return _display(HTML('<div style="display:flex;justify-content:center;">' + html + "</div>"))
    return _display(value)


BOOK_ROOT = Path.cwd().parents[1]
import os
import RiskLabAI
PYTHON_PACKAGE = Path(RiskLabAI.__file__).resolve().parent
PYTHON_SOURCE = PYTHON_PACKAGE.parent
JULIA_ROOT_VALUE = os.environ.get("RISKLABAI_JULIA_ROOT", "")
JULIA_PACKAGE = Path(JULIA_ROOT_VALUE).expanduser() if JULIA_ROOT_VALUE else None
TABLES = BOOK_ROOT / "outputs" / "tables" / "A_risklabai_design"
FIGURES = BOOK_ROOT / "outputs" / "figures" / "A_risklabai_design"
BOOK_FIGURES = FIGURES
FIGURE_DATA = BOOK_ROOT / "outputs" / "figure_data" / "A_risklabai_design"
for directory in [TABLES, FIGURES, BOOK_FIGURES, FIGURE_DATA]:
    directory.mkdir(parents=True, exist_ok=True)

sys.path.insert(0, str(BOOK_ROOT))
from book_figure_style import PALETTE, apply_book_figure_style, finish_axis

apply_book_figure_style()


def rendered_value(value):
    if pd.isna(value):
        return ""
    if isinstance(value, (bool, np.bool_)):
        return "Pass" if bool(value) else "Fail"
    if isinstance(value, (int, np.integer)):
        return f"{int(value):,}"
    if isinstance(value, (float, np.floating)) and float(value).is_integer():
        return f"{int(value):,}"
    return str(value)


def rendered_frame(frame):
    return frame.apply(lambda column: column.map(rendered_value))


def save_table(frame, stem, caption, label, fit=False):
    shown = rendered_frame(frame)
    shown.to_csv(TABLES / f"{stem}.csv", index=False)
    latex = shown.to_latex(index=False, escape=True, caption=caption, label=label, position="htbp")
    if fit:
        latex = latex.replace("\\begin{tabular}", "\\centering\n\\begin{adjustbox}{max width=\\textwidth}\\begin{tabular}", 1).replace("\\end{tabular}", "\\end{tabular}\\end{adjustbox}", 1)
    else:
        latex = latex.replace("\\begin{tabular}", "\\centering\n\\begin{tabular}", 1)
    (TABLES / f"{stem}.tex").write_text(latex, encoding="utf-8")
    display(shown)
    return shown


def save_figure(figure, stem):
    for directory in [FIGURES, BOOK_FIGURES]:
        figure.savefig(directory / f"{stem}.png", dpi=300, bbox_inches="tight", facecolor=PALETTE["white"], transparent=False)
        figure.savefig(directory / f"{stem}.pdf", bbox_inches="tight", facecolor=PALETTE["white"], transparent=False)
    display(figure)
    plt.close(figure)


def add_box(axis, x, y, width, height, text, facecolor, edgecolor, textcolor=None):
    patch = FancyBboxPatch((x - width / 2, y - height / 2), width, height, boxstyle="round,pad=0.04,rounding_size=0.08", facecolor=facecolor, edgecolor=edgecolor, linewidth=1.3, zorder=3)
    axis.add_patch(patch)
    axis.text(x, y, text, ha="center", va="center", color=textcolor or PALETTE["navy"], fontsize=10.2, zorder=4)


import RiskLabAI
from RiskLabAI.core import list_components
from RiskLabAI.core.registry import Registry
assert RiskLabAI.__version__ == "3.2.0"


## Registered names


In [2]:
catalogue = list_components()
registry_rows = []
for family, names in catalogue.items():
    registry_rows.append({"Family": family.replace("_", " ").title(), "Registered components": len(names), "Role": "Built-in catalogue" if names else "Open extension point"})
registry_table = pd.DataFrame(registry_rows)
save_table(registry_table, "python_registry_catalogue", "Registered Python component families in RiskLabAI 3.2.0.", "tab:appendix-python-registry")

local = Registry("affine rules")
local.register("affine", lambda scale=1.0, offset=0.0: {"scale": scale, "offset": offset}, aliases=["linear"], metadata={"purpose": "teaching check"})
built = local.create("linear", scale=2.5, offset=-1.0)
registry_checks = pd.DataFrame([
    {"Check": "Canonical name is listed", "Observed": "affine" in local.available(), "Pass": "affine" in local.available()},
    {"Check": "Alternate name resolves", "Observed": local.aliases().get("linear") == "affine", "Pass": local.aliases().get("linear") == "affine"},
    {"Check": "Construction preserves scale", "Observed": built["scale"], "Pass": built["scale"] == 2.5},
    {"Check": "Construction preserves offset", "Observed": built["offset"], "Pass": built["offset"] == -1.0},
])
save_table(registry_checks, "python_registry_checks", "Observable checks for a small registered component.", "tab:appendix-python-registry-checks", fit=True)


,Family,Registered components,Role
0,Bars,6,Built-in catalogue
1,Cross Validators,6,Built-in catalogue
2,Feature Importance,5,Built-in catalogue
3,Labelers,0,Open extension point
4,Bet Sizers,0,Open extension point
5,Portfolio Optimizers,0,Open extension point


,Check,Observed,Pass
0,Canonical name is listed,Pass,Pass
1,Alternate name resolves,Pass,Pass
2,Construction preserves scale,2.5,Pass
3,Construction preserves offset,-1,Pass


,Check,Observed,Pass
0,Canonical name is listed,Pass,Pass
1,Alternate name resolves,Pass,Pass
2,Construction preserves scale,2.5,Pass
3,Construction preserves offset,-1,Pass


## Shared behavior and compatibility


In [3]:
files = {
    "Registry and construction": PYTHON_PACKAGE / "core" / "registry.py",
    "Common estimator shape": PYTHON_PACKAGE / "core" / "base.py",
    "Replaceable feature method": PYTHON_PACKAGE / "features" / "feature_importance" / "feature_importance_strategy.py",
    "Shared bar workflow": PYTHON_PACKAGE / "data" / "structures" / "abstract_bars.py",
    "Older names remain usable": PYTHON_PACKAGE / "_deprecation.py",
}
design_rows = []
expected_text = {
    "Registry and construction": "class Registry",
    "Common estimator shape": "class Estimator(Protocol)",
    "Replaceable feature method": "class FeatureImportanceStrategy",
    "Shared bar workflow": "class AbstractBars",
    "Older names remain usable": "def deprecated",
}
purposes = {
    "Registry and construction": "Maps stable names to constructors",
    "Common estimator shape": "Accepts objects with fit and predict operations",
    "Replaceable feature method": "Lets one calculation be replaced by another",
    "Shared bar workflow": "Keeps common state while allowing a different stopping rule",
    "Older names remain usable": "Warns before older public names are removed",
}
for choice, path in files.items():
    source = path.read_text(encoding="utf-8")
    design_rows.append({"Design choice": choice, "Public evidence": expected_text[choice], "Purpose": purposes[choice], "Confirmed": expected_text[choice] in source})
design_table = pd.DataFrame(design_rows)
save_table(design_table, "python_component_design", "Confirmed Python design choices in RiskLabAI 3.2.0.", "tab:appendix-python-component-design", fit=True)

stages = pd.DataFrame({"Step": [1, 2, 3, 4], "Action": ["Supply a name", "Normalize the name", "Find the constructor", "Build the component"]})
stages.to_csv(FIGURE_DATA / "python_component_resolution.csv", index=False)
figure, axis = plt.subplots(figsize=(10.5, 3.6), constrained_layout=True)
colors = [PALETTE["near_white"], "#1E3A5F", "#134E4A", "#78350F"]
edges = [PALETTE["gray"], PALETTE["blue"], PALETTE["teal"], PALETTE["orange"]]
for index, row in stages.iterrows():
    x = index * 2.35
    if index < len(stages) - 1:
        axis.add_patch(FancyArrowPatch((x + 0.85, 0), (x + 1.5, 0), arrowstyle="-|>", mutation_scale=15, color=PALETTE["gray"], linewidth=1.4))
    add_box(axis, x, 0, 1.7, 0.9, row["Action"], colors[index], edges[index])
    axis.text(x, 0.72, f"Step {row['Step']}", ha="center", color=PALETTE["gray"], fontsize=9.5)
axis.set(xlim=(-1.1, 8.15), ylim=(-1.0, 1.25), title="From a Public Name to a Constructed Component")
axis.axis("off")
save_figure(figure, "python_component_resolution")


,Design choice,Public evidence,Purpose,Confirmed
0,Registry and construction,class Registry,Maps stable names to constructors,Pass
1,Common estimator shape,class Estimator(Protocol),Accepts objects with fit and predict operations,Pass
2,Replaceable feature method,class FeatureImportanceStrategy,Lets one calculation be replaced by another,Pass
3,Shared bar workflow,class AbstractBars,Keeps common state while allowing a different ...,Pass
4,Older names remain usable,def deprecated,Warns before older public names are removed,Pass


<Figure size 1050x360 with 1 Axes>

## Tests


In [4]:
assert registry_checks["Pass"].all()
assert design_table["Confirmed"].all()
assert len(catalogue["bars"]) == 6
assert len(catalogue["cross_validators"]) == 6
assert len(catalogue["feature_importance"]) == 5
